# 03 — Clean Minimum Wage Data

This notebook processes the FRED state minimum wage history into a clean annual panel that can be
merged with the QCEW employment data in the border-county causal inference analysis.

**Input:** `data/raw/min_wage/state_mw_history.csv`
- Columns: `state` (2-letter abbreviation), `date` (YYYY-MM-DD, monthly from FRED), `min_wage` (float)

**Processing steps:**
1. Parse the date column and extract year.
2. Keep years 2009–2024 (2009 serves as the pre-period baseline).
3. Retain only the **January** observation as the annual minimum wage — standard practice in the
   literature (e.g., Dube, Lester & Reich 2010) because the January value reflects the wage floor
   in effect at the start of the calendar year.
4. Cast `min_wage` to float.
5. Merge with the Census Bureau state FIPS table (`data/raw/state_fips.txt`, downloaded by
   `src/01_download_data.py`) to add a `state_fips` column.

**Output:** `data/intermediate/min_wage_panel.parquet`
- Columns: `state` (2-letter abbr), `state_fips` (2-digit zero-padded string), `year` (int), `min_wage` (float)

In [1]:
import pandas as pd
from pathlib import Path

# ---------------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------------
ROOT = Path().resolve().parent
RAW_MIN_WAGE = ROOT / "data" / "raw" / "min_wage"
INTERMEDIATE = ROOT / "data" / "intermediate"
INTERMEDIATE.mkdir(parents=True, exist_ok=True)
OUTPUT = INTERMEDIATE / "min_wage_panel.parquet"

print("Root         :", ROOT)
print("Raw min wage :", RAW_MIN_WAGE)
print("Output       :", OUTPUT)

Root         : /Users/ogechukwuezenwa/Documents/Duke_University_Projects/Spring_2026/IDS_701-Causal-Inference_Final
Raw min wage : /Users/ogechukwuezenwa/Documents/Duke_University_Projects/Spring_2026/IDS_701-Causal-Inference_Final/data/raw/min_wage
Output       : /Users/ogechukwuezenwa/Documents/Duke_University_Projects/Spring_2026/IDS_701-Causal-Inference_Final/data/intermediate/min_wage_panel.parquet


## 1. Inspect the raw CSV

In [2]:
# Read raw file — keep everything as-is for inspection
raw = pd.read_csv(RAW_MIN_WAGE / "state_mw_history.csv")

print("Shape  :", raw.shape)
print("Dtypes :")
print(raw.dtypes)
print("\nHead:")
raw.head(10)

Shape  : (7796, 3)
Dtypes :
state           str
date            str
min_wage    float64
dtype: object

Head:


,state,date,min_wage
0,AL,1938-10-01,0.25
1,AL,1938-11-01,0.25
2,AL,1938-12-01,0.25
3,AL,1939-01-01,0.25
4,AL,1939-02-01,0.25
5,AL,1939-03-01,0.25
6,AL,1939-04-01,0.25
7,AL,1939-05-01,0.25
8,AL,1939-06-01,0.25
9,AL,1939-07-01,0.25


In [3]:
# Basic descriptive checks on the raw data
print("Unique states :", raw["state"].nunique())
print("States        :", sorted(raw["state"].unique().tolist()))
print("\nDate range    :", raw["date"].min(), "→", raw["date"].max())
print("Min wage range:", raw["min_wage"].min(), "→", raw["min_wage"].max())
print("\nMissing values:")
print(raw.isnull().sum())

Unique states : 51
States        : ['AK', 'AL', 'AR', 'AZ', 'CA', 'CO', 'CT', 'DC', 'DE', 'FL', 'GA', 'HI', 'IA', 'ID', 'IL', 'IN', 'KS', 'KY', 'LA', 'MA', 'MD', 'ME', 'MI', 'MN', 'MO', 'MS', 'MT', 'NC', 'ND', 'NE', 'NH', 'NJ', 'NM', 'NV', 'NY', 'OH', 'OK', 'OR', 'PA', 'RI', 'SC', 'SD', 'TN', 'TX', 'UT', 'VA', 'VT', 'WA', 'WI', 'WV', 'WY']

Date range    : 1938-10-01 → 2026-03-01
Min wage range: 0.25 → 17.95

Missing values:
state       0
date        0
min_wage    0
dtype: int64


## 2. Check monthly structure — multiple entries per state per year

In [ ]:
# Parse date so we can extract year and month
raw["date"] = pd.to_datetime(raw["date"])
raw["year"] = raw["date"].dt.year
raw["month"] = raw["date"].dt.month

# How many observations per state per year?  Should be ~12 for monthly data.
obs_per_state_year = (
    raw.groupby(["state", "year"])["min_wage"].count().rename("n_obs").reset_index()
)

print("Observations per state-year (should be ~12 for monthly FRED series):")
print(obs_per_state_year["n_obs"].value_counts().sort_index())

# Flag any state-years with fewer than 12 observations (may affect edge years)
short = obs_per_state_year[obs_per_state_year["n_obs"] < 12]
if short.empty:
    print("\nAll state-years have 12 observations — clean monthly series.")
else:
    print(f"\nState-years with < 12 observations ({len(short)} cases):")
    print(short.head(20))

Observations per state-year (should be ~12 for monthly FRED series):
n_obs
1     2546
3       10
12     435
Name: count, dtype: int64

State-years with < 12 observations (2556 cases):
   state  year  n_obs
0     AK  1968      1
1     AK  1969      1
2     AK  1970      1
3     AK  1971      1
4     AK  1972      1
5     AK  1973      1
6     AK  1974      1
7     AK  1975      1
8     AK  1976      1
9     AK  1977      1
10    AK  1978      1
11    AK  1979      1
12    AK  1980      1
13    AK  1981      1
14    AK  1982      1
15    AK  1983      1
16    AK  1984      1
17    AK  1985      1
18    AK  1986      1
19    AK  1987      1


## 3. Filter to 2009–2024 and take January values

In [6]:
# Keep only the analysis window (2009 = pre-period baseline, 2024 = last available year)
FIRST_YEAR = 2009
LAST_YEAR = 2024

df = raw[(raw["year"] >= FIRST_YEAR) & (raw["year"] <= LAST_YEAR)].copy()
print(f"Rows after year filter ({FIRST_YEAR}–{LAST_YEAR}): {len(df):,}")

# Retain only the January observation as the annual minimum wage.
# January captures the value in force at the start of the year — the standard
# approach in the literature (Dube, Lester & Reich 2010; Neumark & Wascher 2008).
df_jan = df[df["month"] == 1].copy()
print(f"Rows after keeping January only : {len(df_jan):,}")
print(f"Expected (51 states x 16 years) : {51 * (LAST_YEAR - FIRST_YEAR + 1):,}")

Rows after year filter (2009–2024): 1,694
Rows after keeping January only : 814
Expected (51 states x 16 years) : 816


## 4. Cast types and clean up columns

In [7]:
# Cast min_wage to float (should already be float from CSV, but be explicit)
df_jan["min_wage"] = df_jan["min_wage"].astype(float)

# Cast year to int
df_jan["year"] = df_jan["year"].astype(int)

# Drop helper columns we no longer need
df_jan = df_jan.drop(columns=["date", "month"])

print("Dtypes after cleaning:")
print(df_jan.dtypes)
df_jan.head(10)

Dtypes after cleaning:
state           str
min_wage    float64
year          int64
dtype: object


,state,min_wage,year
843,AL,6.55,2009
855,AL,7.25,2010
867,AL,7.25,2011
879,AL,7.25,2012
891,AL,7.25,2013
903,AL,7.25,2014
915,AL,7.25,2015
927,AL,7.25,2016
939,AL,7.25,2017
951,AL,7.25,2018


## 5. Add state FIPS codes

In [8]:
# Load the Census Bureau state FIPS table downloaded by src/01_download_data.py
# File: data/raw/state_fips.txt  (pipe-delimited, columns: STATE, STUSAB, STATE_NAME, STATENS)
state_fips_df = pd.read_csv(
    ROOT / "data" / "raw" / "state_fips.txt",
    sep="|",
    dtype=str,
)[["STATE", "STUSAB"]].rename(columns={"STATE": "state_fips", "STUSAB": "state"})

# Zero-pad FIPS to 2 digits (Census file already pads, but be explicit)
state_fips_df["state_fips"] = state_fips_df["state_fips"].str.zfill(2)

print(f"Loaded {len(state_fips_df)} states/territories from Census FIPS table")
print(state_fips_df.head())

# Merge FIPS codes onto the panel
df_jan = df_jan.merge(state_fips_df, on="state", how="left")

# Verify — any states that failed to match?
unmatched = df_jan[df_jan["state_fips"].isna()]["state"].unique()
if len(unmatched) == 0:
    print("\nAll states matched to FIPS codes.")
else:
    print(f"\nWARNING: {len(unmatched)} states did not match: {unmatched}")

Loaded 57 states/territories from Census FIPS table
  state_fips state
0         01    AL
1         02    AK
2         04    AZ
3         05    AR
4         06    CA

All states matched to FIPS codes.


## 6. Finalise panel and inspect

In [9]:
# Select and order final columns
panel = df_jan[["state", "state_fips", "year", "min_wage"]].copy()

# Sort for readability
panel = panel.sort_values(["state", "year"]).reset_index(drop=True)

print("=== Final panel ===")
print("Shape  :", panel.shape)
print("Dtypes :")
print(panel.dtypes)
print(f"\nStates : {panel['state'].nunique()} unique")
print(f"Years  : {sorted(panel['year'].unique().tolist())}")
print()
panel.head(20)

=== Final panel ===
Shape  : (814, 4)
Dtypes :
state             str
state_fips        str
year            int64
min_wage      float64
dtype: object

States : 51 unique
Years  : [2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]



,state,state_fips,year,min_wage
0,AK,02,2009,7.15
1,AK,02,2010,7.75
2,AK,02,2011,7.75
3,AK,02,2012,7.75
4,AK,02,2013,7.75
5,AK,02,2014,7.75
6,AK,02,2015,8.75
7,AK,02,2016,9.75
8,AK,02,2017,9.80
9,AK,02,2018,9.84


In [10]:
# Quick summary statistics for min_wage
print("Min wage summary statistics:")
print(panel["min_wage"].describe())

# Check for any suspiciously low or high values
print("\nLowest min wages (spot-check):")
print(
    panel.nsmallest(10, "min_wage")[["state", "year", "min_wage"]].to_string(
        index=False
    )
)

print("\nHighest min wages (spot-check):")
print(
    panel.nlargest(10, "min_wage")[["state", "year", "min_wage"]].to_string(index=False)
)

Min wage summary statistics:
count    814.000000
mean       8.485757
std        2.124122
min        2.650000
25%        7.250000
50%        7.355000
75%        9.250000
max       17.500000
Name: min_wage, dtype: float64

Lowest min wages (spot-check):
state  year  min_wage
   KS  2009      2.65
   GA  2009      5.15
   GA  2010      5.15
   GA  2011      5.15
   GA  2012      5.15
   GA  2013      5.15
   GA  2014      5.15
   GA  2015      5.15
   GA  2016      5.15
   GA  2017      5.15

Highest min wages (spot-check):
state  year  min_wage
   DC  2024     17.50
   DC  2023     16.50
   WA  2024     16.28
   DC  2022     16.10
   CA  2024     16.00
   NY  2024     16.00
   OR  2024     15.95
   WA  2023     15.74
   CT  2024     15.69
   CA  2023     15.50


In [11]:
# Confirm panel is balanced: each state should appear once per year
counts = panel.groupby("state")["year"].count()
expected_years = LAST_YEAR - FIRST_YEAR + 1

if (counts == expected_years).all():
    print(
        f"Panel is balanced: every state has exactly {expected_years} observations (2009–2024)."
    )
else:
    print("WARNING: Panel is unbalanced — some states have unexpected row counts:")
    print(counts[counts != expected_years])

state
GA    15
WY    15
Name: year, dtype: int64


## 7. Save to parquet and verify

In [12]:
# Save as Parquet — compact, typed, and safe for GitHub
panel.to_parquet(OUTPUT, index=False)

print(f"Saved to  : {OUTPUT}")
print(f"File size : {OUTPUT.stat().st_size / 1e3:.1f} KB")
print(f"Rows      : {len(panel):,}")

# Read back to confirm round-trip integrity
check = pd.read_parquet(OUTPUT)
print(f"\nRead back : {check.shape}")
print("Dtypes after round-trip:")
print(check.dtypes)
print("\nSample rows:")
check.sample(8, random_state=42).sort_values(["state", "year"])

Saved to  : /Users/ogechukwuezenwa/Documents/Duke_University_Projects/Spring_2026/IDS_701-Causal-Inference_Final/data/intermediate/min_wage_panel.parquet
File size : 4.7 KB
Rows      : 814

Read back : (814, 4)
Dtypes after round-trip:
state             str
state_fips        str
year            int64
min_wage      float64
dtype: object

Sample rows:


,state,state_fips,year,min_wage
198,IA,19,2016,7.25
227,IL,17,2013,8.25
247,IN,18,2017,7.25
291,LA,22,2013,7.25
294,LA,22,2016,7.25
539,NV,32,2021,9.75
589,OK,40,2023,7.25
800,WY,56,2010,5.15
